In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from pypdf import PdfReader


c:\Users\PatricioGarcia\Desktop\Pato\Programación\Proyectos\NeuroCast_AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# API key
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

# Instance model
openai = OpenAI()

# Model
MODEL = "gpt-4.1-mini"

pdf_text = ""

### Promts definition

In [19]:
summarize_text_promts =  [ 
    { "role": "system",
      "content": "Convertí el contenido en un guion de podcast educativo. "
                  "Debe sonar natural, claro y conversacional, como un profesor explicando. "
                  "No leas literalmente el PDF."
                  "No indiques nada sobre la música" },
     ]

make_script_promts =  [ 
    { "role": "system",
      "content": "Sos un asistente de estudio. Resumí el contenido de forma clara, breve y útil para aprender. Devolve solo el resumen, no hagas aclaraciones" },
     ]

chat_system_prompt = "Sos un profesor particular que a partir de un resumen hace preguntas para ayudar a un estudiante a aprender sobre el siguiente tema: "

### Pipeline

In [20]:
# PDF reader definition
def read_pdf(pdf_file):

    reader = PdfReader(pdf_file.name)
    text = []

    for page in reader.pages:
        content = page.extract_text()
        if content:
            text.append(content)
    return "\n\n".join(text)

# Make summary
def summarize_text(pdf_text):

    user_prompt = {"role": "user",
                   "content": f"Resumí este material:\n\n{pdf_text}" }
    summarize_text_promts.append(user_prompt)

    response = openai.chat.completions.create( 
                                                model=MODEL,
                                                messages=summarize_text_promts
    )

    return response.choices[0].message.content

# Make podcast script
def make_script(summary):

    user_prompt = {"role": "user",
                   "content": f"Transformá este contenido en un podcast de estudio:\n\n{summary}" }
    make_script_promts.append(user_prompt)

    response = openai.chat.completions.create( 
                                                model=MODEL,
                                                messages=make_script_promts
    )

    return response.choices[0].message.content

# Transform to voice
def talker(script):

    output_path = r'C:\Users\PatricioGarcia\Desktop\Pato\Programación\Proyectos\NeuroCast_AI\podcast.mp3'

    response = openai.audio.speech.create(
        model = 'gpt-4o-mini-tts',
        voice = 'onyx',
        input=script
    )

    response.stream_to_file(output_path)
    return str(output_path)


# Orchestrate PDF processing
def process_pdf(pdf_file):
    pdf_text = read_pdf(pdf_file) 
    summary = summarize_text(pdf_text)
    script = make_script(summary)
    audio_path = talker(script)

    # Save process state
    state = {
        "pdf_text" : pdf_text,
        "summary" : summary,
        "script" : script
    }
    
    return summary, audio_path, state



In [16]:
def chat(message, history, state):

    # If not pdf loaded
    if not state or not state.get("pdf_text"):
        history = history or [] # If none, to list
        history.append({ "role": "user", "content": message })
        history.append({ "role": "assistant", "content": "Primero subí y procesá un PDF para que pueda ayudarte a estudiarlo." })
        return history

    # PDF loaded
    history = history or [] # If none, to list
    system_prompt = f"""
        Sos un tutor de estudio.
        Respondé únicamente en base al siguiente material.

        Resumen:
        {state['summary']}

        Contenido del podcast:
        {state['script']}

        Si algo no está en el material, decilo claramente.
        """
    # Add system promt to messages
    messages = [{"role": "system", "content": system_prompt}]
    
    # Format history and add history to messages
    for h in history:
        messages.append({"role": h["role"], "content": h["content"]})

    # Add user message to messages
    messages.append( {"role": "user", "content": message} )

    # Query the bot
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    reply = response.choices[0].message.content

    # Save new user message and reply in history
    history.append( {"role": "user", "content": message} )
    history.append( {"role": "assistant", "content": reply})

    return history, ""

In [ ]:
with gr.Blocks() as ui:
    state = gr.State({})

    with gr.Row():
        pdf_input = gr.File(file_types=[".pdf"], label="Drop PDF file")
        process_btn = gr.Button("Procesar PDF")

    with gr.Row():
        summary_output = gr.Markdown(label="Resumen")
        audio_output = gr.Audio(label="Audio output", autoplay=False, interactive=False)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("")

        with gr.Column(scale=1):
            chatbot = gr.Chatbot(height=400)
            message_box = gr.Textbox(label="Preguntale al bot sobre el PDF")
            send_btn = gr.Button("Enviar")

    process_btn.click(
        fn=process_pdf,
        inputs=pdf_input,
        outputs=[summary_output, audio_output, state]
    )

    send_btn.click(
        fn=chat,
        inputs=[message_box, chatbot, state],
        outputs=[chatbot, message_box]
    )

    message_box.submit(
        fn=chat,
        inputs=[message_box, chatbot, state],
        outputs=[chatbot, message_box]
    )

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


C:\Users\PatricioGarcia\AppData\Local\Temp\ipykernel_35868\1738901578.py:52: DeprecationWarning: Due to a bug, this method doesn't actually stream the response content, `.with_streaming_response.method()` should be used instead
  response.stream_to_file(output_path)
Exception in callback _ProactorBasePipeTransport._call_connection_lost(None)
handle: <Handle _ProactorBasePipeTransport._call_connection_lost(None)>
Traceback (most recent call last):
  File "C:\Users\PatricioGarcia\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
  File "C:\Users\PatricioGarcia\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\asyncio\proactor_events.py", line 165, in _call_connection_lost
    self._sock.shutdown(socket.SHUT_RDWR)
ConnectionResetError: [WinError 10054] Se ha forzado la interrupción de una conexión existente por el host remoto
